[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C65_ProblemSolving_Communication_Course/03_tradeoffs/03_tradeoffs_decisions.ipynb)

# 03 · 权衡与决策（加权打分 / 权重扰动敏感性分析 / 帕累托前沿 / 贪心vs背包 / 不确定性决策准则 / 沉没成本）

目标：把「选 A 还是选 B」从一次拍脑袋，变成一套**可以运行、可以断言的决策方法论**。

本 notebook 你会亲手实现：
1. **多维加权打分器** —— 列维度、定权重、归一化、打分排序
2. **权重扰动敏感性分析** —— 权重扰动会不会翻转结论，这是本模块最有说服力的一步
3. **帕累托前沿计算** —— 不需要权重就能淘汰掉「被支配」的方案
4. **预算约束下的方案组合** —— 贪心 vs 0/1 背包最优，量化贪心到底差多少
5. **三种不确定性决策准则** —— 期望效用 / 最坏情况 / 后悔最小化在同一问题上的分歧
6. **可逆决策速度与沉没成本模拟** —— 用代码证明"已经投入多少"不该影响面向未来的选择

> 心智模型：**权衡题考的不是选哪个，是你选择背后那条推理链能不能被摊开来审查。**

## 1 · 多维加权打分器：列维度 → 定权重 → 打分

三个改进方案在四个维度上的原始打分（1-10，越高越好），按权重加权求和排序。

In [ ]:
def weighted_score(scores, weights):
    """score = sum(w_i * s_i)，要求各维度已经统一量纲（见下面的归一化演示）。"""
    return sum(s * w for s, w in zip(scores, weights))

def rank_options(options, weights):
    """options: {方案名: [各维度打分, ...]}  ->  (按分数降序的名字列表, {名字: 分数})"""
    scored = {name: weighted_score(scores, weights) for name, scores in options.items()}
    return sorted(scored, key=lambda k: -scored[k]), scored

DIMENSIONS = ['准确率提升', '实现成本(已反向,越高越省事)', '维护复杂度(已反向,越高越省事)', '延迟影响(已反向,越高越省事)']
OPTIONS = {
    '方案A：升级 backbone':   [9, 3, 4, 5],
    '方案B：加一层后处理规则': [5, 8, 7, 8],
    '方案C：扩充训练数据':     [6, 6, 6, 6],
}
WEIGHTS = [0.4, 0.2, 0.15, 0.25]
assert abs(sum(WEIGHTS) - 1.0) < 1e-9

order, scored = rank_options(OPTIONS, WEIGHTS)
for name in order:
    print(f'{name:<24} {scored[name]:.2f} 分')

assert order[0] == '方案B：加一层后处理规则'
assert abs(scored['方案B：加一层后处理规则'] - 6.65) < 1e-9
assert abs(scored['方案A：升级 backbone'] - 6.05) < 1e-9
assert abs(scored['方案C：扩充训练数据'] - 6.00) < 1e-9
print(f"\n✅ 方案B领先方案A {scored['方案B：加一层后处理规则']-scored['方案A：升级 backbone']:.2f} 分"
      f"，领先方案C {scored['方案B：加一层后处理规则']-scored['方案C：扩充训练数据']:.2f} 分——")
print('   但这个领先幅度稳不稳健，取决于权重本身有多可信，这正是下一节要检验的。')

## 2 · 归一化：不同量纲的分数不能直接相加

如果原始打分不是统一的 1-10 量表，而是"延迟(ms，越小越好)"和"准确率(%，越大越好)"这类真实单位，
必须先做最小-最大归一化，再套加权公式。

In [ ]:
def minmax_normalize(raw_values, higher_is_better=True):
    """把一组原始数值归一化到 [0,1]。higher_is_better=False 时先取负数再归一化（越小原始值归一化后越接近1）。"""
    vals = [v if higher_is_better else -v for v in raw_values]
    lo, hi = min(vals), max(vals)
    if hi == lo:
        return [0.5] * len(vals)          # 所有候选在这个维度上并列，归一化给中性值
    return [(v - lo) / (hi - lo) for v in vals]

# 三个方案的"真实单位"表现：延迟(ms，越小越好)，准确率提升(百分点，越大越好)
latency_ms = [12, 3, 7]          # 方案A/B/C
acc_gain_pp = [4.5, 1.2, 2.8]

norm_latency = minmax_normalize(latency_ms, higher_is_better=False)
norm_acc = minmax_normalize(acc_gain_pp, higher_is_better=True)

print('原始延迟(ms)      :', latency_ms, ' -> 归一化(越大越好):', [round(x, 3) for x in norm_latency])
print('原始准确率提升(pp):', acc_gain_pp, ' -> 归一化(越大越好):', [round(x, 3) for x in norm_acc])

assert norm_latency[1] == 1.0 and norm_latency[0] == 0.0   # 延迟最小(方案B,3ms) 归一化到1；最大(方案A,12ms) 归一化到0
assert norm_acc[0] == 1.0 and norm_acc[1] == 0.0            # 准确率提升最大(方案A) 归一化到1；最小(方案B) 归一化到0
print('\n✅ 归一化之后，延迟和准确率提升才能在同一个 [0,1] 量纲上被加权相加，直接把 12ms 和 4.5pp 相加是没有意义的。')

## 3 · 权重扰动敏感性分析：结论稳不稳健

在原始权重附近按相对比例随机扰动、重新归一化、重新打分排序，重复几千次，
统计"最优方案被翻转"的比例——这是本模块最有说服力的一步。

In [ ]:
import random

def perturb_weights(weights, rng, scale=0.3):
    """每个权重按 ±scale 的相对比例随机浮动，再重新归一化使权重和为 1。"""
    noisy = [max(0.01, w * (1 + rng.uniform(-scale, scale))) for w in weights]
    total = sum(noisy)
    return [w / total for w in noisy]

def sensitivity_analysis(options, base_weights, rng, n_trials=2000, scale=0.3):
    """返回 (原始最优方案, 翻转率)。翻转率 = 扰动后最优方案与原始不同的试验占比。"""
    base_top = rank_options(options, base_weights)[0][0]
    flips = 0
    for _ in range(n_trials):
        w = perturb_weights(base_weights, rng, scale)
        top = rank_options(options, w)[0][0]
        if top != base_top:
            flips += 1
    return base_top, flips / n_trials

rng = random.Random(0)
top, flip_rate = sensitivity_analysis(OPTIONS, WEIGHTS, rng, n_trials=2000, scale=0.3)
print(f'原始最优方案：{top}')
print(f'±30% 权重扰动下，翻转率 = {flip_rate:.3f}（{flip_rate:.1%}）')
assert top == '方案B：加一层后处理规则'
assert abs(flip_rate - 0.045) < 1e-9
print('\n✅ 4.5% 的翻转率说明结论总体稳健，但不是绝对稳健——面试里可以诚实地报这个数字，而不是笼统地说"肯定选B"。')

In [ ]:
# 对照组：方案在每个维度都不劣于其他方案时（帕累托支配），翻转率应该精确为 0——
# 这是下一节"被支配"概念的一个预告，也验证了"支配关系 -> 加权分数必然占优"这个数学事实。
ROBUST_OPTIONS = {
    'E（全维度占优）': [9, 9, 9, 9],
    'F': [5, 5, 5, 5],
    'G': [3, 3, 3, 3],
}
rng2 = random.Random(1)
top2, flip_rate2 = sensitivity_analysis(ROBUST_OPTIONS, WEIGHTS, rng2, n_trials=2000, scale=0.3)
print(f'全维度占优方案：{top2}，翻转率 = {flip_rate2}')
assert top2 == 'E（全维度占优）'
assert flip_rate2 == 0.0

# 再看翻转率如何随扰动幅度增大而上升——扰动越大，越容易触达"决策边界"
print(f"\n{'扰动幅度':>8} {'翻转率':>8}")
prev_rate = -1.0
for scale in (0.1, 0.2, 0.3, 0.5, 0.8):
    rng3 = random.Random(0)
    _, rate = sensitivity_analysis(OPTIONS, WEIGHTS, rng3, n_trials=2000, scale=scale)
    print(f'{scale:>8.1f} {rate:>8.3f}')
    assert rate >= prev_rate, '扰动幅度增大，翻转率不应该下降'
    prev_rate = rate
print('\n✅ 翻转率随扰动幅度单调不减：这正是"结论对权重的依赖程度"可以被量化观察的证据。')

## 4 · 帕累托前沿：不需要权重就能淘汰的方案

方案 c 和方案 b 成本相同但收益更低、方案 e 比方案 d 更贵却收益相同——这两个可以无条件淘汰，
不需要争论任何权重。

In [ ]:
CANDIDATES = {
    'a': {'cost': 1, 'benefit': 2},
    'b': {'cost': 2, 'benefit': 3},
    'c': {'cost': 2, 'benefit': 2},
    'd': {'cost': 3, 'benefit': 5},
    'e': {'cost': 4, 'benefit': 5},
    'f': {'cost': 5, 'benefit': 6},
}

def is_dominated(name, candidates):
    """存在另一个候选，成本不高于它、收益不低于它、且至少一项严格更优 -> 被支配。"""
    x = candidates[name]
    for other, y in candidates.items():
        if other == name:
            continue
        if y['cost'] <= x['cost'] and y['benefit'] >= x['benefit'] and (y['cost'] < x['cost'] or y['benefit'] > x['benefit']):
            return True
    return False

def pareto_front(candidates):
    return sorted([n for n in candidates if not is_dominated(n, candidates)], key=lambda n: candidates[n]['cost'])

front = pareto_front(CANDIDATES)
dominated = [n for n in CANDIDATES if n not in front]
print('帕累托前沿：', front)
print('被支配（可直接淘汰）：', dominated)

assert front == ['a', 'b', 'd', 'f']
assert dominated == ['c', 'e']
print('\n✅ c 被 b 支配（成本相同收益更低）、e 被 d 支配（收益相同成本更高）——')
print('   这一步筛选完全不需要定权重，是加权打分之前免费的第一道过滤（呼应 C53-05 的三步选型法）。')

## 5 · 预算约束下的方案组合：贪心先跑一遍

三个候选修复方案 (成本, 收益)，预算 50。先看贪心按性价比排序能拿到多少——
`knapsack_best`（0/1 背包精确解）留在练习 3，你自己实现后再和贪心的结果对比。

In [ ]:
ITEMS = [('X', 10, 60), ('Y', 20, 100), ('Z', 30, 120)]   # (名字, 成本, 收益)
BUDGET = 50

def greedy_by_ratio(items, budget):
    """按 收益/成本 比降序贪心装入，直到预算装不下下一个为止。"""
    order = sorted(items, key=lambda it: -it[2] / it[1])
    chosen, cost, value = [], 0, 0
    for name, c, v in order:
        if cost + c <= budget:
            chosen.append(name)
            cost += c
            value += v
    return chosen, cost, value

g_chosen, g_cost, g_value = greedy_by_ratio(ITEMS, BUDGET)
ratios = sorted([(n, round(v / c, 2)) for n, c, v in ITEMS], key=lambda t: -t[1])
print('性价比排序（收益/成本，降序）：', ratios)
print(f'贪心选择: {g_chosen}  成本={g_cost}  收益={g_value}')

assert g_chosen == ['X', 'Y']
assert g_cost == 30 and g_value == 160
print('\n贪心用了 30/50 预算，拿到 160 收益——看起来不错，但预算还剩 20 没用完。')
print('练习 3 会让你亲手写 0/1 背包 DP，验证贪心到底是不是真的最优。')

## 6 · 三种不确定性决策准则的分歧演示

要不要为极端天气单独训练一个专用小模型？常见天气发生概率 90%，极端天气 10%。
三种决策准则可能给出不同答案——这个分歧本身就是本节最重要的结论。

In [ ]:
SCENARIO_PROB = {'常见天气': 0.9, '极端天气': 0.1}
PAYOFF = {
    '专用模型': {'常见天气': 6, '极端天气': 9},
    '通用模型': {'常见天气': 8, '极端天气': 3},
    '不做':     {'常见天气': 9, '极端天气': 0},
}

def expected_utility(payoff, probs):
    return {a: sum(probs[s] * v[s] for s in probs) for a, v in payoff.items()}

def maximin(payoff):
    return {a: min(v.values()) for a, v in payoff.items()}

def minimax_regret(payoff):
    scenarios = list(next(iter(payoff.values())).keys())
    best_per_scenario = {s: max(payoff[a][s] for a in payoff) for s in scenarios}
    return {a: max(best_per_scenario[s] - payoff[a][s] for s in scenarios) for a in payoff}

eu = expected_utility(PAYOFF, SCENARIO_PROB)
mm = maximin(PAYOFF)
mr = minimax_regret(PAYOFF)

best_eu = max(eu, key=eu.get)
best_mm = max(mm, key=mm.get)
best_mr = min(mr, key=mr.get)

print('期望效用   :', {k: round(v, 2) for k, v in eu.items()}, '-> 选', best_eu)
print('最坏情况   :', mm, '-> 选', best_mm)
print('后悔最小化 :', mr, '-> 选', best_mr)

assert best_eu == '不做'
assert best_mm == '专用模型'
assert best_mr == '专用模型'
print('\n✅ 期望效用选"不做"（长期概率占优），最坏情况和后悔最小化都选"专用模型"（防极端场景兜底）——')
print('   三选二地分裂：这说明这不是一道靠直觉能一步到位的题，需要先确认团队对风险的容忍态度。')

## 7 · 可逆决策速度与沉没成本谬误

In [ ]:
def decision_speed_budget(reversibility, stakes):
    """reversibility: 'reversible'/'irreversible'；stakes: 1-10（影响范围/撤回代价）。
    返回"建议投入的决策精力"相对值——不可逆决策在同等 stakes 下应该分配数倍精力。"""
    base = {'reversible': 1, 'irreversible': 4}[reversibility]
    return base * stakes

DECISIONS = [
    ('调整置信度阈值(线上可回滚)', 'reversible', 2),
    ('更换标注供应商', 'reversible', 3),
    ('删除历史训练数据快照', 'irreversible', 7),
    ('选择车端推理芯片平台', 'irreversible', 9),
]
for name, kind, stakes in DECISIONS:
    budget = decision_speed_budget(kind, stakes)
    print(f'{name:<24} {kind:<12} stakes={stakes}  建议决策精力(相对值)={budget}')

assert decision_speed_budget('reversible', 5) < decision_speed_budget('irreversible', 5)
print('\n✅ 同等影响范围下，不可逆决策应该分配数倍的决策精力——这不是"性格谨慎"，是显式的速度分配规则。')

In [ ]:
# 沉没成本谬误：理性决策只看未来价值，不应该被"已经投入多少"左右
def rational_choice(future_value_current, future_value_alternative, sunk_cost):
    return 'switch' if future_value_alternative > future_value_current else 'stay'

def biased_choice(future_value_current, future_value_alternative, sunk_cost, sunk_weight=0.1):
    """沉没成本谬误：把已投入的沉没成本按比例折算进"继续当前方案"的分数里。"""
    adjusted_current = future_value_current + sunk_weight * sunk_cost
    return 'switch' if future_value_alternative > adjusted_current else 'stay'

# 同样的"未来价值对比"（新架构未来价值8 明显高于旧架构未来价值5），只是已投入的沉没成本不同
assert rational_choice(future_value_current=5, future_value_alternative=8, sunk_cost=0) == 'switch'
assert rational_choice(future_value_current=5, future_value_alternative=8, sunk_cost=1000) == 'switch'
print('理性决策：不管已经投入多少（0 还是 1000），结论都是 switch —— 未来价值对比没有变。')

assert biased_choice(future_value_current=5, future_value_alternative=8, sunk_cost=0) == 'switch'
assert biased_choice(future_value_current=5, future_value_alternative=8, sunk_cost=1000) == 'stay'
print('偏误决策：沉没成本从 0 涨到 1000 之后，结论从 switch 被拖成了 stay —— 这正是"都已经投入这么久了"背后的谬误。')
print('\n✅ 理性决策函数对沉没成本的大小完全不敏感；偏误决策函数会被拖着不肯换——这就是可以在面试里讲清楚的机制。')

## ✏️ 练习 1：多维打分的归一化 + 加权排序一体化

实现 `score_options(raw_options, directions, weights)`：`raw_options` 是 `{方案名: [原始值,...]}`，
`directions` 是每个维度 `'max'`（越大越好）或 `'min'`（越小越好）的列表。
函数需要先对每个维度做最小-最大归一化（复用第 2 节的 `minmax_normalize`），再加权求和，
返回 `{方案名: 加权总分}`。

In [ ]:
def score_options(raw_options, directions, weights):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
raw = {
    '方案A': [12, 4.5],   # [延迟ms, 准确率提升pp]
    '方案B': [3, 1.2],
    '方案C': [7, 2.8],
}
directions = ['min', 'max']       # 延迟越小越好，准确率提升越大越好
w = [0.5, 0.5]
result = score_options(raw, directions, w)
print(result)

assert abs(result['方案A'] - (0.0 * 0.5 + 1.0 * 0.5)) < 1e-9     # 延迟最差(归一化0)，准确率提升最好(归一化1)
assert abs(result['方案B'] - (1.0 * 0.5 + 0.0 * 0.5)) < 1e-9     # 延迟最好(归一化1)，准确率提升最差(归一化0)
assert 0.0 < result['方案C'] < 1.0                                 # 方案C两项都居中
print('✅ 练习 1 通过：先归一化到同一量纲，再加权，才能公平地合成一个数字。')

## ✏️ 练习 2：n 维帕累托前沿

实现 `pareto_front_nd(candidates, directions)`：泛化第 4 节的二维版本到任意维度。
`candidates`: `{名字: [维度值,...]}`；`directions`: 每维 `'max'` 或 `'min'`。
返回未被支配的名字列表（被支配定义：存在另一候选在每一维都不差、且至少一维更好）。

In [ ]:
def pareto_front_nd(candidates, directions):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def _better_or_equal(v, u, directions):
    """v 在每一维是否都不差于 u（按 directions 的方向）。"""
    for vi, ui, d in zip(v, u, directions):
        if d == 'max' and vi < ui:
            return False
        if d == 'min' and vi > ui:
            return False
    return True

# 三维候选：(成本[越小越好], 收益[越大越好], 风险[越小越好])
CANDS_3D = {
    'p1': [1, 2, 5],
    'p2': [2, 3, 3],
    'p3': [2, 2, 3],   # 与 p2 成本相同，收益更低、风险相同 -> 被 p2 支配
    'p4': [3, 5, 2],
    'p5': [4, 5, 4],   # 与 p4 相比：成本更高、收益相同、风险更高 -> 被 p4 支配
}
front3d = pareto_front_nd(CANDS_3D, ['min', 'max', 'min'])
print('三维帕累托前沿：', sorted(front3d))

assert sorted(front3d) == ['p1', 'p2', 'p4']
assert 'p3' not in front3d and 'p5' not in front3d

# 退化检验：全部候选互不支配时，前沿应该等于候选全集
CANDS_TIE = {'x': [1, 5], 'y': [2, 6], 'z': [3, 7]}   # 成本越高收益也越高，谁都不支配谁
front_tie = pareto_front_nd(CANDS_TIE, ['min', 'max'])
assert sorted(front_tie) == ['x', 'y', 'z']
print('✅ 练习 2 通过：n 维支配关系的判断和二维完全一样，只是要在每一维上都检查。')

## ✏️ 练习 3：0/1 背包动态规划（验证贪心到底差多少）

实现 `knapsack_best(items, budget)`：`items` 是 `[(名字, 成本, 收益), ...]`，成本均为非负整数。
返回 `(选中的名字列表, 最优总收益)`。用动态规划求**精确最优解**，再和第 5 节的贪心结果对比。

In [ ]:
def knapsack_best(items, budget):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
k_chosen, k_value = knapsack_best(ITEMS, BUDGET)
print(f'背包最优解: {sorted(k_chosen)}  收益={k_value}')

assert sorted(k_chosen) == ['Y', 'Z']
assert k_value == 220
assert k_value > g_value, '最优解不应该比贪心解差'
print(f'\n对比：贪心收益 {g_value}（用了 {g_cost}/{BUDGET} 预算），背包最优 {k_value}（用满 50/{BUDGET} 预算）')
print(f'贪心比最优少了 {k_value - g_value} 收益，相对差距 {(k_value - g_value) / k_value:.1%}——')
print('✅ 练习 3 通过：这正是"性价比最高不等于组合起来最优"的量化证据。')

## ✏️ 练习 4：统一的不确定性决策器

实现 `decide_under_uncertainty(payoff, probs, criterion)`：`criterion` ∈
`'expected_utility' / 'maximin' / 'minimax_regret'`，返回该准则下的最优方案名字。
复用第 6 节已经定义的 `expected_utility` / `maximin` / `minimax_regret` 三个函数即可，
本练习只是把"选择哪个准则"这件事封装成一个统一入口。

In [ ]:
def decide_under_uncertainty(payoff, probs, criterion):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert decide_under_uncertainty(PAYOFF, SCENARIO_PROB, 'expected_utility') == '不做'
assert decide_under_uncertainty(PAYOFF, SCENARIO_PROB, 'maximin') == '专用模型'
assert decide_under_uncertainty(PAYOFF, SCENARIO_PROB, 'minimax_regret') == '专用模型'

try:
    decide_under_uncertainty(PAYOFF, SCENARIO_PROB, 'unknown_criterion')
    assert False, '未知准则应该报错'
except (KeyError, ValueError):
    pass
print('✅ 练习 4 通过：三种准则封装成一个入口后，面试里可以现场把同一个 payoff 表在三种准则下各跑一遍。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def score_options(raw_options, directions, weights):
    names = list(raw_options)
    n_dims = len(directions)
    norm_cols = []
    for d in range(n_dims):
        col = [raw_options[name][d] for name in names]
        higher_is_better = (directions[d] == 'max')
        norm_cols.append(minmax_normalize(col, higher_is_better=higher_is_better))
    result = {}
    for i, name in enumerate(names):
        result[name] = sum(norm_cols[d][i] * weights[d] for d in range(n_dims))
    return result

In [ ]:
# 练习 2 参考答案
def pareto_front_nd(candidates, directions):
    def dominated(name):
        v = candidates[name]
        for other, u in candidates.items():
            if other == name:
                continue
            if _better_or_equal(u, v, directions) and u != v:
                return True
        return False
    return [n for n in candidates if not dominated(n)]

In [ ]:
# 练习 3 参考答案
def knapsack_best(items, budget):
    n = len(items)
    dp = [[0] * (budget + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        name, c, v = items[i - 1]
        for b in range(budget + 1):
            dp[i][b] = dp[i - 1][b]
            if c <= b:
                dp[i][b] = max(dp[i][b], dp[i - 1][b - c] + v)
    b = budget
    chosen = []
    for i in range(n, 0, -1):
        if dp[i][b] != dp[i - 1][b]:
            name, c, v = items[i - 1]
            chosen.append(name)
            b -= c
    return list(reversed(chosen)), dp[n][budget]

In [ ]:
# 练习 4 参考答案
def decide_under_uncertainty(payoff, probs, criterion):
    if criterion == 'expected_utility':
        eu = expected_utility(payoff, probs)
        return max(eu, key=eu.get)
    if criterion == 'maximin':
        mm = maximin(payoff)
        return max(mm, key=mm.get)
    if criterion == 'minimax_regret':
        mr = minimax_regret(payoff)
        return min(mr, key=mr.get)
    raise ValueError(f'未知准则: {criterion}')

---
## 🧪 真实工程胶囊：权衡决策白板模板 + 面试口播要点

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 权衡决策白板模板（面试当场可以照这个骨架展开）
# ══════════════════════════════════════════════════════════════════════
# 1. 列维度：这个决策涉及哪几个互相独立的考量？有没有维度在重复计分？
# 2. 定权重：权重来自业务/安全优先级，独立于候选方案打分之前先定好
# 3. 打分：不同量纲先做最小-最大归一化，再加权求和
# 4. 帕累托预筛：有没有方案在所有维度都被别的方案碾压？先淘汰，不用权重
# 5. 敏感性分析：把权重扰动 ±20%~30%，结论会不会翻转？翻转率多高？
# 6. 不确定性：如果各方案的表现依赖未来场景，明确说清楚用的是期望效用/最坏情况/后悔最小化中的哪个
# 7. 可逆性：这个决策能不能轻松撤回？决定该用多快的速度、要不要多方会签
# 8. 说清放弃了什么：选中的方案在哪个维度不是最优，代价是什么，什么条件下会重新评估

# ══════════════════════════════════════════════════════════════════════
# B. 面试里最容易被追问的三句话，提前想好怎么答
# ══════════════════════════════════════════════════════════════════════
# Q: "你为什么选 A 不选 B？"
# A: "我列了四个维度、按这样定权重，A 领先 B 0.6 分；我把权重扰动了 ±30% 重跑了一遍，
#     A 领先的结论在 95% 的情况下依然成立，所以这个结论是稳健的。"
#
# Q: "那 B 更好的地方怎么办？"
# A: "B 在实现成本上确实更低，选 A 意味着我们要多付出这部分工程代价；
#     如果后续发现这个代价超出预算，我会重新评估。"
#
# Q: "如果只能选一个，你选哪个？"
# A: "我选 A。我的判断标准是安全 > 用户体验 > 开发效率，这个场景涉及漏检安全类别，
#     所以按这个标准 A 优先；这意味着我们暂时不改善另一类的误检率。"

# ══════════════════════════════════════════════════════════════════════
# C. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 帕累托前沿与延迟-精度权衡的技术细节        -> C53 模块 05（本课只讲面试组织表达）
# · 预算约束下贪心与背包最优的技术细节          -> C57 模块 05（本课复用其两步法结论）
# · 诊断出根因之后的方案选择                    -> 承接 C65 模块 02
# · 模糊问题的收敛与优先级排序                  -> C65 模块 04（"只能选一个"框架与之衔接）
'''
print(RECIPE)
for token in ['帕累托预筛', '敏感性分析', '不确定性', 'C53 模块 05', 'C57 模块 05']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：权衡决策白板骨架 / 高频追问的标准答法 / 与其他课程的分工边界')

### 小结

- **取舍显式化的四步是列维度→定权重→打分→说清放弃了什么**——最后一步几乎总被省略，但正是它把"选择"变成"决策"。
- **权重扰动敏感性分析是本模块最有说服力的一步**：与其笼统地说"选 A"，不如报出"权重扰动 30% 下翻转率 4.5%"，
  用一个数字证明结论的稳健程度；全维度占优的方案，翻转率精确为 0，这是可以严格证明的数学事实。
- **帕累托前沿不需要权重就能淘汰方案**：被支配的方案无论权重怎么定都不该选，这是加权打分之前免费的第一道过滤。
- **性价比最高不等于组合起来最优**：0/1 背包问题里贪心没有最坏情况保证，候选数量少时应该用 DP 精确验证。
- **不确定性下的三种决策准则会给出不同答案**：期望效用适合可重复、概率靠谱的场景；最坏情况和后悔最小化
  适合后果极端或不可挽回的场景——分歧本身就是值得主动指出的洞察。
- **可逆决策快做，不可逆决策慢做**；沉没成本不该影响面向未来的选择，理性决策函数对已投入多少完全不敏感。
- **"如果只能选一个"必须给出明确选择 + 独立准则 + 放弃的代价**——"都很重要"式回答是最危险的失分项。

下一站：**模块 04 · 模糊问题与需求澄清** —— 权衡的前提是候选方案已经摆在桌上，
下一个更难的问题是：题目本身连边界都没有，你要先把它收敛成一个能权衡的问题。